## Baking subreddit data cleaning and enriching

This notebook cleans the raw r/baking data, inclduing:
- Dropping unneeded columns
- Converting the post date from UTC (ms since epoch) to MM/DD/YYYY
- Doing a fuzzy match to filter for posts that are similar to GBBO technical bakes
- Calculating and adding a column that denotes the distance from the relevant GBBO episode in which the post was made.

In [1]:
import os
import json
import glob
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz, utils
from pathlib import Path
from datetime import datetime, timezone
import re

In [2]:
data_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/baking_reddit_data_raw.csv")
if data_csv.is_file():
    baking_reddit_df = pd.read_csv(data_csv)
    print(f"file already exists")
else:
    data=[]
    with open ('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/r_baking_posts.jsonl', 'r') as file:
        for line in file:
            data.append(json.loads(line))
        baking_reddit_df = pd.read_json("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/r_baking_posts.jsonl", lines=True)

file already exists


In [3]:
columns_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/baking_reddit_raw_data_columns.csv")
if columns_csv.is_file():
    print(f"file already exists")
else:
    columns = baking_reddit_df.columns.to_frame(index=False)
    columns.to_csv('baking_reddit_raw_data_columns.csv', index=False)

file already exists


In [4]:
baking_reddit_df

,Unnamed: 0,created_utc,thumbnail,title,url
0,0,1280630410,NaN,"Come to Cupcake Camp OC, tomorrow. Eat tons of...",http://i2.photobucket.com/albums/y6/Eunmi429/c...
1,1,1280944111,NaN,My junior's cheesecake woes... any advice appr...,http://imgur.com/hBJlw
2,2,1280949404,default,Chocolate Cake with Chocolate Peanut Butter Fr...,http://blog.junbelen.com/2010/02/07/how-to-mak...
3,3,1280952218,default,First Attempt at White Bread,https://www.reddit.com/r/Baking/comments/cxey3...
4,4,1280985772,self,HELP -- Pastry Cream Butter Cream is BROKEN,https://www.reddit.com/r/Baking/comments/cxl68...
...,...,...,...,...,...
528981,528981,1784675977,https://preview.redd.it/h4f9b1gy1oeh1.jpg?widt...,Self Developed Recipe! Walnut Shortbread brown...,https://www.reddit.com/gallery/1v2ypjc
528982,528982,1784676571,default,Biscuits à la Sally,NaN
528983,528983,1784676698,https://preview.redd.it/99ko9rg34oeh1.jpeg?wid...,Biscuits à la Sally,https://i.redd.it/99ko9rg34oeh1.jpeg
528984,528984,1784676765,https://preview.redd.it/qj3eujca4oeh1.jpg?widt...,"Low k screwed up the lattice , but it still lo...",https://www.reddit.com/gallery/1v2z0qo


In [5]:
baking_reddit_df_cols_to_keep = [
    'created_utc',
    'thumbnail',
    'title',
    'url',
]

baking_reddit_df_clean = baking_reddit_df[baking_reddit_df_cols_to_keep]
baking_reddit_df_clean.to_csv('baking_reddit_data_complete.csv')
baking_reddit_df_clean

,created_utc,thumbnail,title,url
0,1280630410,NaN,"Come to Cupcake Camp OC, tomorrow. Eat tons of...",http://i2.photobucket.com/albums/y6/Eunmi429/c...
1,1280944111,NaN,My junior's cheesecake woes... any advice appr...,http://imgur.com/hBJlw
2,1280949404,default,Chocolate Cake with Chocolate Peanut Butter Fr...,http://blog.junbelen.com/2010/02/07/how-to-mak...
3,1280952218,default,First Attempt at White Bread,https://www.reddit.com/r/Baking/comments/cxey3...
4,1280985772,self,HELP -- Pastry Cream Butter Cream is BROKEN,https://www.reddit.com/r/Baking/comments/cxl68...
...,...,...,...,...
528981,1784675977,https://preview.redd.it/h4f9b1gy1oeh1.jpg?widt...,Self Developed Recipe! Walnut Shortbread brown...,https://www.reddit.com/gallery/1v2ypjc
528982,1784676571,default,Biscuits à la Sally,NaN
528983,1784676698,https://preview.redd.it/99ko9rg34oeh1.jpeg?wid...,Biscuits à la Sally,https://i.redd.it/99ko9rg34oeh1.jpeg
528984,1784676765,https://preview.redd.it/qj3eujca4oeh1.jpg?widt...,"Low k screwed up the lattice , but it still lo...",https://www.reddit.com/gallery/1v2z0qo


In [6]:
baking_reddit_df_clean['date'] = pd.to_datetime(baking_reddit_df_clean['created_utc'], unit='s')
baking_reddit_df_clean['date'] = baking_reddit_df_clean['date'].dt.strftime('%m/%d/%Y')
baking_reddit_df_clean

,created_utc,thumbnail,title,url,date
0,1280630410,NaN,"Come to Cupcake Camp OC, tomorrow. Eat tons of...",http://i2.photobucket.com/albums/y6/Eunmi429/c...,08/01/2010
1,1280944111,NaN,My junior's cheesecake woes... any advice appr...,http://imgur.com/hBJlw,08/04/2010
2,1280949404,default,Chocolate Cake with Chocolate Peanut Butter Fr...,http://blog.junbelen.com/2010/02/07/how-to-mak...,08/04/2010
3,1280952218,default,First Attempt at White Bread,https://www.reddit.com/r/Baking/comments/cxey3...,08/04/2010
4,1280985772,self,HELP -- Pastry Cream Butter Cream is BROKEN,https://www.reddit.com/r/Baking/comments/cxl68...,08/05/2010
...,...,...,...,...,...
528981,1784675977,https://preview.redd.it/h4f9b1gy1oeh1.jpg?widt...,Self Developed Recipe! Walnut Shortbread brown...,https://www.reddit.com/gallery/1v2ypjc,07/21/2026
528982,1784676571,default,Biscuits à la Sally,NaN,07/21/2026
528983,1784676698,https://preview.redd.it/99ko9rg34oeh1.jpeg?wid...,Biscuits à la Sally,https://i.redd.it/99ko9rg34oeh1.jpeg,07/21/2026
528984,1784676765,https://preview.redd.it/qj3eujca4oeh1.jpg?widt...,"Low k screwed up the lattice , but it still lo...",https://www.reddit.com/gallery/1v2z0qo,07/21/2026


In [7]:
baking_reddit_post_titles = baking_reddit_df_clean['title']
baking_reddit_post_titles

0         Come to Cupcake Camp OC, tomorrow. Eat tons of...
1         My junior's cheesecake woes... any advice appr...
2         Chocolate Cake with Chocolate Peanut Butter Fr...
3                              First Attempt at White Bread
4               HELP -- Pastry Cream Butter Cream is BROKEN
                                ...                        
528981    Self Developed Recipe! Walnut Shortbread brown...
528982                                  Biscuits à la Sally
528983                                  Biscuits à la Sally
528984    Low k screwed up the lattice , but it still lo...
528985    Please Share Your Best Banana Bread Recipe As ...
Name: title, Length: 528986, dtype: str

In [8]:
technicals_df = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/TechnicalBakes.csv')
technicals_cols = technicals_df['Technical']
technicals_cols

0                       victoria sandwich
1                                  scones
2                                     cob
3                 mini hot lemon soufflés
4                         cornish pasties
                      ...                
129          lemon and thyme drizzle cake
130    orange and ginger treacle puddings
131                      caterpiller cake
132                       tart aux pommes
133                     lardy cake slices
Name: Technical, Length: 134, dtype: str

In [9]:
#iterate through each technical
matched_csv = Path("/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/baking_reddit_posts_matched.csv")

if matched_csv.is_file():
    baking_reddit_df = pd.read_csv(matched_csv)
    print(f"file already exists")

else:

    choices = technicals_cols
    titles = baking_reddit_df_clean['title'].tolist()

    score_matrix = process.cdist(titles, choices, scorer=fuzz.ratio)

    best_scores = score_matrix.max(axis=1)
    best_idx = score_matrix.argmax(axis=1)

    baking_reddit_df_clean['bakeoff_score'] = best_scores
    baking_reddit_df_clean['bakeoff_match'] = [choices[i] for i in best_idx]
    baking_reddit_df_clean['bakeoff'] = best_scores > 78

    baking_reddit_df_clean = baking_reddit_df_clean[baking_reddit_df_clean['bakeoff']][['date','title', 'bakeoff_match', 'bakeoff_score']]
    baking_reddit_df_clean.to_csv('baking_reddit_posts_matched.csv')

In [10]:
#add air dates
technical_bakes = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/TechnicalBakes.csv')

baking_reddit_df_clean_dated = pd.merge(baking_reddit_df_clean, technical_bakes, left_on='bakeoff_match', right_on='Technical', how='left')
baking_reddit_df_clean_dated


,date,title,bakeoff_match,bakeoff_score,Unnamed: 0,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric
0,04/12/2011,Red velvet cake?,red velvet cake,90.322578,114,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0
1,07/20/2011,I made English muffins!!,english muffins,80.000000,25,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN
2,01/15/2012,Chocolate cake,chocolate teacakes,78.787880,21,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0
3,01/15/2012,Chocolate cake,chocolate teacakes,78.787880,21,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0
4,02/19/2012,Red Velvet Cake,red velvet cake,80.000000,114,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1182,07/14/2026,English Muffins!,english muffins,81.250000,25,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN
1183,07/14/2026,Sticky toffee pudding,sticky toffee puddings,90.909088,107,10/12/2021,12,4,desserts,pavlova,165.0,sticky toffee puddings,90.0,joconde imprime dessert,270.0,2.0
1184,07/15/2026,Chocolate cakes,chocolate teacakes,82.352943,21,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0
1185,07/17/2026,Scones,scones,83.333336,1,8/24/2010,1,2,biscuit,personality biscuits,120.0,scones,60.0,"meringue, choux, and macaron petits fours",240.0,1.0


In [11]:
baking_reddit_df_clean_dated.drop(columns=['Unnamed: 0', 'bakeoff_match'])
baking_reddit_df_clean_dated['reddit_post_title'] = baking_reddit_df_clean_dated['title']
baking_reddit_df_clean_dated['reddit_post_date'] = baking_reddit_df_clean_dated['date']
baking_reddit_df_clean_dated

,date,title,bakeoff_match,bakeoff_score,Unnamed: 0,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric,reddit_post_title,reddit_post_date
0,04/12/2011,Red velvet cake?,red velvet cake,90.322578,114,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red velvet cake?,04/12/2011
1,07/20/2011,I made English muffins!!,english muffins,80.000000,25,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,I made English muffins!!,07/20/2011
2,01/15/2012,Chocolate cake,chocolate teacakes,78.787880,21,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cake,01/15/2012
3,01/15/2012,Chocolate cake,chocolate teacakes,78.787880,21,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cake,01/15/2012
4,02/19/2012,Red Velvet Cake,red velvet cake,80.000000,114,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red Velvet Cake,02/19/2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1182,07/14/2026,English Muffins!,english muffins,81.250000,25,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,English Muffins!,07/14/2026
1183,07/14/2026,Sticky toffee pudding,sticky toffee puddings,90.909088,107,10/12/2021,12,4,desserts,pavlova,165.0,sticky toffee puddings,90.0,joconde imprime dessert,270.0,2.0,Sticky toffee pudding,07/14/2026
1184,07/15/2026,Chocolate cakes,chocolate teacakes,82.352943,21,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cakes,07/15/2026
1185,07/17/2026,Scones,scones,83.333336,1,8/24/2010,1,2,biscuit,personality biscuits,120.0,scones,60.0,"meringue, choux, and macaron petits fours",240.0,1.0,Scones,07/17/2026


In [12]:
GBBO_reddit_posts_merged = baking_reddit_df_clean_dated.drop(columns=['title', 'date', 'Unnamed: 0'])
GBBO_reddit_posts_merged

,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric,reddit_post_title,reddit_post_date
0,red velvet cake,90.322578,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red velvet cake?,04/12/2011
1,english muffins,80.000000,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,I made English muffins!!,07/20/2011
2,chocolate teacakes,78.787880,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cake,01/15/2012
3,chocolate teacakes,78.787880,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cake,01/15/2012
4,red velvet cake,80.000000,9/13/2022,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red Velvet Cake,02/19/2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1182,english muffins,81.250000,8/27/2013,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,English Muffins!,07/14/2026
1183,sticky toffee puddings,90.909088,10/12/2021,12,4,desserts,pavlova,165.0,sticky toffee puddings,90.0,joconde imprime dessert,270.0,2.0,Sticky toffee pudding,07/14/2026
1184,chocolate teacakes,82.352943,10/2/2012,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cakes,07/15/2026
1185,scones,83.333336,8/24/2010,1,2,biscuit,personality biscuits,120.0,scones,60.0,"meringue, choux, and macaron petits fours",240.0,1.0,Scones,07/17/2026


In [16]:
#calculating days since technical airdate in which reddit post was made

GBBO_reddit_posts_merged['Airdate'] = pd.to_datetime(GBBO_reddit_posts_merged['Airdate'])
GBBO_reddit_posts_merged['reddit_post_date'] = pd.to_datetime(GBBO_reddit_posts_merged['reddit_post_date'])

# GBBO_reddit_posts_merged['days_since_air'] = []

posts = GBBO_reddit_posts_merged['reddit_post_title']

for post in posts:
    GBBO_reddit_posts_merged['days_since_air'] = GBBO_reddit_posts_merged['Airdate'] - GBBO_reddit_posts_merged['reddit_post_date']

GBBO_reddit_posts_merged

,bakeoff_match,bakeoff_score,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),challenge level numeric,reddit_post_title,reddit_post_date,days_since_air
0,red velvet cake,90.322578,2022-09-13,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red velvet cake?,2011-04-12,4172 days
1,english muffins,80.000000,2013-08-27,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,I made English muffins!!,2011-07-20,769 days
2,chocolate teacakes,78.787880,2012-10-02,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cake,2012-01-15,261 days
3,chocolate teacakes,78.787880,2012-10-02,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cake,2012-01-15,261 days
4,red velvet cake,80.000000,2022-09-13,13,1,cake,12 mini sandwich cakes,120.0,red velvet cake,120.0,3d cake home,240.0,2.0,Red Velvet Cake,2012-02-19,3859 days
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1182,english muffins,81.250000,2013-08-27,4,2,bread,36 breadsticks,120.0,english muffins,165.0,decorative loaf,240.0,NaN,English Muffins!,2026-07-14,-4704 days
1183,sticky toffee puddings,90.909088,2021-10-12,12,4,desserts,pavlova,165.0,sticky toffee puddings,90.0,joconde imprime dessert,270.0,2.0,Sticky toffee pudding,2026-07-14,-1736 days
1184,chocolate teacakes,82.352943,2012-10-02,3,8,biscuit,48 crackers,120.0,chocolate teacakes,120.0,gingerbread structure,240.0,2.0,Chocolate cakes,2026-07-15,-5034 days
1185,scones,83.333336,2010-08-24,1,2,biscuit,personality biscuits,120.0,scones,60.0,"meringue, choux, and macaron petits fours",240.0,1.0,Scones,2026-07-17,-5806 days


In [17]:
GBBO_reddit_posts_merged.to_csv('GBBO_reddit_posts_merged.csv')